# Multimodal Cardiac Fusion Model v3 — ECG + Echocardiogram
**Task:** Dual binary classification — structural disease + arrhythmia detection  
**Architecture:** Learned ECG Tokenizer + ResNet50 (frozen) + Task-Aware Cross-Attention Fusion + Dual MLP Heads  
**Training:** End-to-end with partial freeze (ResNet50 locked, ECG encoder trainable)  
**XAI:** Electromechanical Attribution Maps (EAM) + Confidence Calibration  

### v3 Changes from v2
1. **Learned ECG Tokenizer** — replaces naive equal-thirds P/QRS/T split with soft attention that learns which temporal regions to group
2. **Task-Aware Pooling** — structural head emphasises echo tokens, arrhythmia head emphasises ECG tokens (learned, not hardcoded)
3. **Partial Freeze** — ResNet50 stays frozen (ImageNet weights preserved), only ECG encoder + fusion + heads train (~4M params instead of 27M)
4. **Confidence Calibration** — temperature scaling + reliability diagrams for clinically trustworthy probability estimates

## 0 · Environment Check
Verify GPU availability and VRAM. Requires CUDA GPU with ≥6 GB VRAM.

In [ ]:
import torch

print(f"PyTorch   : {torch.__version__}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {props.total_memory / 1e9:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {DEVICE}")

## 1 · Configuration

**Multi-task setup:**
- Two binary tasks: **structural disease** (normal vs disease) and **arrhythmia** (normal rhythm vs arrhythmia)
- `LAMBDA_STRUCT = 1.0` (primary), `LAMBDA_ARR = 0.5` (auxiliary — forces ECG encoder to learn useful features)

**v3 architecture changes:**
- `NUM_ECG_TOKENS = 4` — learned tokenizer produces 4 soft tokens via attention (replaces hardcoded 3 equal thirds)
- `PROJ_DIM = 96` — shared embedding space, kept small to prevent overfitting on ~3200 samples
- ResNet50 is **frozen** — only ~4M parameters train (vs 27M in v2)

**Training:**
- No Stage 1 — direct end-to-end with partial freeze
- `weight_decay = 1e-2` — strong L2 regularisation to combat overfitting
- `label_smoothing = 0.1` — prevents overconfident predictions
- Early stopping on combined macro-F1 (average of structural + arrhythmia)

In [ ]:
import os, random
import numpy as np
import torch

# ── Paths ─────────────────────────────────────────────────────────────────────
CACHE_DIR  = r"C:\Users\anwme\Desktop\Datasets\cache"
OUTPUT_DIR = r"C:\Users\anwme\Desktop\Datasets\model_output_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────────
ECG_LEADS   = 12
ECG_LEN     = 5000
ECHO_SIZE   = 224
ECHO_FRAMES = 3

# ── Task ──────────────────────────────────────────────────────────────────────
NUM_CLASSES_STRUCT = 2
NUM_CLASSES_ARR    = 2
STRUCT_CLASSES     = ['normal', 'structural_disease']
ARR_CLASSES        = ['normal_rhythm', 'arrhythmia']

LAMBDA_STRUCT = 1.0
LAMBDA_ARR    = 0.5

# ── Model ─────────────────────────────────────────────────────────────────────
ECG_BASE_DIM  = 256
ECHO_BASE_DIM = 2048
PROJ_DIM      = 96
NUM_ECG_TOKENS = 4      # NEW: learned tokenizer produces 4 tokens (was 3 hardcoded)
ATTN_HEADS    = 4
ATTN_LAYERS   = 2
ATTN_DROPOUT  = 0.1
MLP_DROPOUT   = 0.5

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 16
S2_EPOCHS  = 50         # more room to converge with fewer trainable params
S2_LR      = 5e-5       # slightly higher since fewer params
PATIENCE   = 12
SEED       = 42

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device     : {DEVICE}")
print(f"ECG tokens : {NUM_ECG_TOKENS} (learned)")
print(f"PROJ_DIM   : {PROJ_DIM}")
print(f"Config OK")

## 2 · Dataset & DataLoaders

**Label mapping (binary for both tasks):**
- **Structural:** `normal` → 0, all disease subtypes → 1
- **Arrhythmia:** `normal` → 0, all arrhythmia subtypes → 1

**Each sample returns 4 tensors:** ECG `(12, 5000)`, Echo `(3, 224, 224)`, structural label, arrhythmia label

**Class imbalance handling:**
- `WeightedRandomSampler` oversamples structurally-normal patients 4× during training
- Focal loss + class weights handle remaining imbalance (see Section 4)

**Augmentation (training only):**
- ECG: amplitude jitter (±20%), Gaussian noise, time shift (up to 1s), random lead dropout
- Echo: horizontal flip, brightness/contrast jitter

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

with open(os.path.join(CACHE_DIR, 'classes.json')) as f:
    meta = json.load(f)
OLD_CLASSES = meta['classes']
print(f"Original 6 classes: {OLD_CLASSES}")
print(f"Binary structural : {STRUCT_CLASSES}")
print(f"Binary arrhythmia : {ARR_CLASSES}")


class CardiacDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        data         = np.load(npz_path, mmap_mode='r')
        self.ecg     = data['ecg']
        self.echo    = data['echo']
        self.augment = augment
        self.echo_mean = torch.tensor([0.449], dtype=torch.float32)
        self.echo_std  = torch.tensor([0.226], dtype=torch.float32)

        self.labels_struct = data['label_struct_bin'].copy()
        self.labels_arr    = data['label_arr_bin'].copy()

        ns = np.bincount(self.labels_struct, minlength=2)
        na = np.bincount(self.labels_arr, minlength=2)
        print(f"  Structural: normal={ns[0]}  disease={ns[1]}  ratio=1:{ns[1]/max(ns[0],1):.1f}")
        print(f"  Arrhythmia: normal={na[0]}  arrhythmia={na[1]}  ratio=1:{na[1]/max(na[0],1):.1f}")

    def __len__(self):
        return len(self.labels_struct)

    def _augment_ecg(self, ecg):
        ecg = ecg * np.random.uniform(0.8, 1.2)
        ecg = ecg + np.random.randn(*ecg.shape).astype(np.float32) * 0.05
        shift = np.random.randint(0, 500)
        ecg = np.concatenate([ecg[:, shift:],
                               np.zeros((ecg.shape[0], shift), dtype=np.float32)], axis=1)
        if np.random.rand() > 0.5:
            n_drop = np.random.randint(1, 3)
            drop_idx = np.random.choice(12, n_drop, replace=False)
            ecg[drop_idx] = 0
        return ecg

    def _augment_echo(self, echo):
        if np.random.rand() > 0.5:
            echo = echo[:, :, ::-1].copy()
        if np.random.rand() > 0.5:
            gain = np.random.uniform(0.8, 1.2)
            bias = np.random.uniform(-15, 15)
            echo = np.clip(echo.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)
        return echo

    def __getitem__(self, idx):
        ecg  = self.ecg[idx].copy().astype(np.float32)
        echo = self.echo[idx].copy()
        lbl_s = int(self.labels_struct[idx])
        lbl_a = int(self.labels_arr[idx])

        ecg = np.nan_to_num(ecg, nan=0.0, posinf=8.0, neginf=-8.0)
        ecg = np.clip(ecg, -8.0, 8.0)

        if self.augment:
            ecg  = self._augment_ecg(ecg)
            echo = self._augment_echo(echo)

        ecg_t  = torch.from_numpy(ecg)
        echo_t = torch.from_numpy(echo.astype(np.float32) / 255.0)
        echo_t = (echo_t - self.echo_mean.view(1, 1, 1)) / self.echo_std.view(1, 1, 1)

        return (ecg_t, echo_t,
                torch.tensor(lbl_s, dtype=torch.long),
                torch.tensor(lbl_a, dtype=torch.long))


train_ds = CardiacDataset(os.path.join(CACHE_DIR, 'train.npz'), augment=True)
val_ds   = CardiacDataset(os.path.join(CACHE_DIR, 'val.npz'),   augment=False)
test_ds  = CardiacDataset(os.path.join(CACHE_DIR, 'test.npz'),  augment=False)

sample_weights = np.where(train_ds.labels_struct == 0, 4.0, 1.0)
sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"\nTrain : {len(train_ds):>5d}")
print(f"Val   : {len(val_ds):>5d}")
print(f"Test  : {len(test_ds):>5d}")

## 3 · Model Architecture — v3 Dual-Head with Learned Tokenizer + Task-Aware Pooling

```
ECG  (B, 12, 5000)                     Echo (B, 3, 224, 224)
       |                                        |
  [1D-ResNet]                         [ResNet50 FROZEN]
  8 ResBlocks                          shared across 3 frames
  output: (B, 256, ~156)               output: (B, 3, 2048)
       |                                        |
 [Learned Tokenizer]  ← NEW            [Cycle Tokenizer]
  soft attention over time              project each frame
  output: (B, 4, 96)                   output: (B, 3, 96)
       |  + learnable pos emb          |  + learnable pos emb
       +──────── [Cross-Attention ×2] ─+
                 bidirectional ECG ↔ Echo
                 output: (B, 7, 96)
                          |
              [Task-Aware Pooling]  ← NEW
          learned attention per task
          structural → emphasises echo tokens
          arrhythmia → emphasises ECG tokens
               |                |
        [Struct Head]    [Arr Head]
         256→64→2         256→64→2
            |                |
     Normal vs Disease   Normal vs Arrhythmia
```

### v3 Changes

**1. Learned ECG Tokenizer (replaces PhaseTokenizer)**  
The v2 tokenizer split the ECG feature map into 3 equal thirds and called them P/QRS/T. But a 10-second recording contains ~10-15 heartbeats — equal thirds don't correspond to any physiological phases. The new tokenizer uses a 1×1 convolution to produce `NUM_ECG_TOKENS` (4) soft attention maps over time, then pools the feature map through those maps. The model learns which temporal regions to group together — no hardcoded assumptions.

**2. Task-Aware Pooling (replaces flat concatenation)**  
In v2, both heads received the same flattened fusion vector. The structural head had no reason to focus on echo tokens, and the arrhythmia head had no reason to focus on ECG tokens. Task-aware pooling adds a small learned attention layer per task that weights the 7 fused tokens differently. The structural head automatically learns to up-weight echo tokens; the arrhythmia head up-weights ECG tokens.

**3. ResNet50 FROZEN**  
v2 unfroze all 27.5M parameters and overfitted (98% train, 73% val). v3 freezes ResNet50 (~23.5M params), leaving only ~4M trainable. The ECG encoder trains from scratch (it needs to learn cardiac features), while ResNet50's ImageNet features are good enough for echo frame analysis.

**4. 4 ECG tokens + 3 echo tokens = 7 total**  
The cross-attention now operates on 4 ECG × 3 echo tokens, giving a richer interaction matrix for the EAM visualisation.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


# ── A. ECG ENCODER — 1D ResNet ───────────────────────────────────────────────
# Input : (B, 12, 5000)
# Output: (B, 256, ~156)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 7, stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 7, padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.skip(x))


class ECGEncoder(nn.Module):
    def __init__(self, in_leads=12, base_dim=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_leads, 64, 15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(3, stride=2, padding=1)
        )
        self.layer1 = self._make(64,  64,       n=2, stride=1)
        self.layer2 = self._make(64,  128,      n=2, stride=2)
        self.layer3 = self._make(128, 192,      n=2, stride=2)
        self.layer4 = self._make(192, base_dim, n=2, stride=2)

    def _make(self, in_ch, out_ch, n, stride):
        return nn.Sequential(
            ResBlock1D(in_ch, out_ch, stride=stride),
            *[ResBlock1D(out_ch, out_ch) for _ in range(n - 1)]
        )

    def forward(self, x):
        return self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))


# ── B. LEARNED ECG TOKENIZER (NEW — replaces hardcoded P/QRS/T split) ────────
# Uses soft attention to learn which temporal regions to group into tokens.
# Input : (B, C, T) from ECG encoder
# Output: (B, num_tokens, proj_dim)

class LearnedPhaseTokenizer(nn.Module):
    def __init__(self, in_dim=256, proj_dim=96, num_tokens=4):
        super().__init__()
        self.num_tokens = num_tokens
        self.token_proj = nn.Conv1d(in_dim, num_tokens, kernel_size=1)
        self.proj = nn.Linear(in_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)

    def forward(self, x):  # x: (B, C, T)
        weights = self.token_proj(x).softmax(dim=-1)      # (B, num_tokens, T)
        tokens = torch.bmm(weights, x.transpose(1, 2))    # (B, num_tokens, C)
        return self.norm(self.proj(tokens))                # (B, num_tokens, proj_dim)


# ── C. ECHO ENCODER — ResNet50 (will be FROZEN) ──────────────────────────────
# Input : (B, 3, 224, 224)
# Output: (B, 3, 2048)

class EchoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        bb = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        old = bb.conv1
        new = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        new.weight.data = old.weight.data.mean(dim=1, keepdim=True)
        bb.conv1 = new
        self.backbone = nn.Sequential(*list(bb.children())[:-1])

    def forward(self, echo):
        B, F, H, W = echo.shape
        x = echo.reshape(B * F, 1, H, W)
        x = self.backbone(x).flatten(1)
        return x.reshape(B, F, -1)


# ── D. CYCLE TOKENIZER ───────────────────────────────────────────────────────
# Input : (B, 3, 2048)
# Output: (B, 3, proj_dim)

class CycleTokenizer(nn.Module):
    def __init__(self, in_dim=2048, proj_dim=96):
        super().__init__()
        self.proj = nn.Linear(in_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)

    def forward(self, x):
        return self.norm(self.proj(x))


# ── E. CROSS-ATTENTION LAYER ─────────────────────────────────────────────────
# Bidirectional: ECG queries → Echo keys AND Echo queries → ECG keys
# Handles asymmetric token counts (4 ECG × 3 echo)

class CrossAttentionLayer(nn.Module):
    def __init__(self, dim=96, num_heads=4, dropout=0.1):
        super().__init__()
        self.ecg_to_echo = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.echo_to_ecg = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_ecg    = nn.LayerNorm(dim)
        self.norm_echo   = nn.LayerNorm(dim)
        self.attn_weights = {}

    def forward(self, ecg_tok, echo_tok):
        ecg_att,  w_e2o = self.ecg_to_echo(ecg_tok,  echo_tok, echo_tok)
        echo_att, w_o2e = self.echo_to_ecg(echo_tok, ecg_tok,  ecg_tok)
        ecg_out  = self.norm_ecg( ecg_tok  + ecg_att)
        echo_out = self.norm_echo(echo_tok + echo_att)
        self.attn_weights = {
            'ecg_to_echo': w_e2o.detach().cpu(),
            'echo_to_ecg': w_o2e.detach().cpu()
        }
        return ecg_out, echo_out


# ── F. STACKED CROSS-ATTENTION FUSION ────────────────────────────────────────
# Returns fused tokens (B, num_ecg+3, dim) WITHOUT flattening — needed for task-aware pooling

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=96, num_heads=4, dropout=0.1, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            CrossAttentionLayer(dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

    @property
    def attn_weights(self):
        return self.layers[-1].attn_weights

    def forward(self, ecg_tok, echo_tok):
        for layer in self.layers:
            ecg_tok, echo_tok = layer(ecg_tok, echo_tok)
        return torch.cat([ecg_tok, echo_tok], dim=1)  # (B, N_ecg+3, dim)


# ── G. TASK-AWARE POOLING (NEW) ──────────────────────────────────────────────
# Each task learns which tokens matter most via a small attention layer.
# Structural naturally learns to weight echo tokens higher.
# Arrhythmia naturally learns to weight ECG tokens higher.

class TaskAwarePooling(nn.Module):
    def __init__(self, dim=96):
        super().__init__()
        self.struct_attn = nn.Linear(dim, 1)
        self.arr_attn    = nn.Linear(dim, 1)

    def forward(self, fused_tokens):  # (B, N_total, dim)
        s_w = self.struct_attn(fused_tokens).softmax(dim=1)  # (B, N, 1)
        s_pooled = (fused_tokens * s_w).sum(dim=1)           # (B, dim)

        a_w = self.arr_attn(fused_tokens).softmax(dim=1)
        a_pooled = (fused_tokens * a_w).sum(dim=1)

        return s_pooled, a_pooled, s_w.squeeze(-1), a_w.squeeze(-1)


# ── H. FULL FUSION MODEL — v3 ────────────────────────────────────────────────

class CardiacFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.ecg_encoder  = ECGEncoder(in_leads=ECG_LEADS, base_dim=ECG_BASE_DIM)
        self.echo_encoder = EchoEncoder()
        self.phase_tok    = LearnedPhaseTokenizer(in_dim=ECG_BASE_DIM, proj_dim=PROJ_DIM,
                                                  num_tokens=NUM_ECG_TOKENS)
        self.cycle_tok    = CycleTokenizer(in_dim=ECHO_BASE_DIM, proj_dim=PROJ_DIM)
        self.ecg_pos  = nn.Parameter(torch.randn(1, NUM_ECG_TOKENS, PROJ_DIM) * 0.02)
        self.echo_pos = nn.Parameter(torch.randn(1, 3, PROJ_DIM) * 0.02)
        self.fusion   = CrossAttentionFusion(PROJ_DIM, ATTN_HEADS, ATTN_DROPOUT, ATTN_LAYERS)
        self.task_pool = TaskAwarePooling(dim=PROJ_DIM)

        self.head_structural = nn.Sequential(
            nn.Linear(PROJ_DIM, 256), nn.GELU(), nn.Dropout(MLP_DROPOUT),
            nn.Linear(256, 64),       nn.GELU(), nn.Dropout(MLP_DROPOUT / 2),
            nn.Linear(64, NUM_CLASSES_STRUCT)
        )
        self.head_arrhythmia = nn.Sequential(
            nn.Linear(PROJ_DIM, 256), nn.GELU(), nn.Dropout(MLP_DROPOUT),
            nn.Linear(256, 64),       nn.GELU(), nn.Dropout(MLP_DROPOUT / 2),
            nn.Linear(64, NUM_CLASSES_ARR)
        )

    def forward(self, ecg, echo):
        ecg_feat  = self.ecg_encoder(ecg)                      # (B, 256, ~156)
        echo_feat = self.echo_encoder(echo)                    # (B, 3, 2048)
        ecg_tok   = self.phase_tok(ecg_feat)  + self.ecg_pos   # (B, 4, 96)
        echo_tok  = self.cycle_tok(echo_feat) + self.echo_pos  # (B, 3, 96)
        fused     = self.fusion(ecg_tok, echo_tok)             # (B, 7, 96)
        s_pooled, a_pooled, _, _ = self.task_pool(fused)       # (B, 96) each
        return self.head_structural(s_pooled), self.head_arrhythmia(a_pooled)

    def freeze_echo_encoder(self):
        for p in self.echo_encoder.parameters():
            p.requires_grad = False
        print("Echo encoder (ResNet50) FROZEN.")

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True
        print("All layers UNFROZEN.")

    def count_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen    = total - trainable
        print(f"Total params    : {total/1e6:.2f} M")
        print(f"Trainable       : {trainable/1e6:.2f} M")
        print(f"Frozen          : {frozen/1e6:.2f} M")


# ── Instantiate & dry-run ─────────────────────────────────────────────────────
model = CardiacFusionModel().to(DEVICE)
model.freeze_echo_encoder()
model.count_params()

with torch.no_grad():
    s_out, a_out = model(torch.randn(2, 12, 5000).to(DEVICE),
                         torch.randn(2, 3, 224, 224).to(DEVICE))
print(f"\nDry-run: structural={s_out.shape}  arrhythmia={a_out.shape}")
print(f"NaN check: {not (torch.isnan(s_out).any() or torch.isnan(a_out).any())}")

## 4 · Training Utilities

**Focal Loss** — down-weights easy, correctly-classified examples (mostly majority class) and focuses on hard, misclassified ones (mostly minority class). `γ=2.0` is the standard setting from RetinaNet.

**Multi-task training loop** — each batch computes `total_loss = 1.0 × loss_structural + 0.5 × loss_arrhythmia`. The 0.5 weight means arrhythmia acts as a regulariser that guides the ECG encoder without dominating training.

**Model selection** — best checkpoint saved by combined macro-F1 (average of structural and arrhythmia macro-F1). This ensures neither task is neglected.

In [ ]:
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score)


def compute_class_weights_binary(labels, num_classes=2):
    counts  = np.bincount(labels, minlength=num_classes).astype(float)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls    = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def run_epoch_mt(model, loader, crit_s, crit_a, optimizer=None, scaler=None, use_mixup=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = s_correct = a_correct = total = 0
    use_amp = DEVICE.type == 'cuda'

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for ecg, echo, lbl_s, lbl_a in loader:
            ecg, echo = ecg.to(DEVICE), echo.to(DEVICE)
            lbl_s, lbl_a = lbl_s.to(DEVICE), lbl_a.to(DEVICE)

            if is_train: optimizer.zero_grad()

            if use_amp:
                with torch.autocast('cuda', torch.float16):
                    out_s, out_a = model(ecg, echo)
                    loss = LAMBDA_STRUCT * crit_s(out_s, lbl_s) + LAMBDA_ARR * crit_a(out_a, lbl_a)
            else:
                out_s, out_a = model(ecg, echo)
                loss = LAMBDA_STRUCT * crit_s(out_s, lbl_s) + LAMBDA_ARR * crit_a(out_a, lbl_a)

            if torch.isnan(loss) or torch.isinf(loss):
                if is_train: optimizer.zero_grad()
                continue

            if is_train:
                if use_amp and scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            total_loss += loss.item() * lbl_s.size(0)
            s_correct  += (out_s.argmax(1) == lbl_s).sum().item()
            a_correct  += (out_a.argmax(1) == lbl_a).sum().item()
            total      += lbl_s.size(0)

    if total == 0: return float('nan'), 0.0, 0.0
    return total_loss / total, s_correct / total, a_correct / total


def evaluate_mt(model, loader):
    model.eval()
    s_probs, a_probs, s_true, a_true = [], [], [], []
    use_amp = DEVICE.type == 'cuda'
    with torch.no_grad():
        for ecg, echo, lbl_s, lbl_a in loader:
            if use_amp:
                with torch.autocast('cuda', torch.float16):
                    out_s, out_a = model(ecg.to(DEVICE), echo.to(DEVICE))
            else:
                out_s, out_a = model(ecg.to(DEVICE), echo.to(DEVICE))
            s_probs.extend(torch.softmax(out_s, 1)[:, 1].cpu().tolist())
            a_probs.extend(torch.softmax(out_a, 1)[:, 1].cpu().tolist())
            s_true.extend(lbl_s.tolist())
            a_true.extend(lbl_a.tolist())
    return (np.array(s_true), np.array(s_probs),
            np.array(a_true), np.array(a_probs))


def compute_combined_f1(model, loader):
    s_true, s_probs, a_true, a_probs = evaluate_mt(model, loader)
    s_f1 = f1_score(s_true, (s_probs >= 0.5).astype(int), average='macro', zero_division=0)
    a_f1 = f1_score(a_true, (a_probs >= 0.5).astype(int), average='macro', zero_division=0)
    return s_f1, a_f1, (s_f1 + a_f1) / 2


def plot_history_mt(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title(f'{title} — Loss'); axes[0].legend()
    axes[1].plot(history['train_s_acc'], label='Train')
    axes[1].plot(history['val_s_acc'], label='Val')
    axes[1].set_title(f'{title} — Structural Acc'); axes[1].legend()
    axes[2].plot(history['train_a_acc'], label='Train')
    axes[2].plot(history['val_a_acc'], label='Val')
    axes[2].set_title(f'{title} — Arrhythmia Acc'); axes[2].legend()
    for ax in axes: ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'history_{title}.png'), dpi=150)
    plt.show()


def plot_confusion(true_labels, pred_labels, class_names, title):
    cm = confusion_matrix(true_labels, pred_labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'cm_{title}.png'), dpi=150)
    plt.show()


print("Utilities ready.")

## 5 · Model Initialisation (Stage 1 Skipped)

Frozen-encoder training (Stage 1) consistently plateaued at macro-F1 ≈ 0.22 in v2 experiments. The pretrained ResNet50 and randomly-initialised ECG encoder don't produce features discriminative enough for the fusion head to learn from in isolation.

**v3 approach — partial freeze:**
- ResNet50 (echo encoder) stays frozen — its ImageNet features are good enough for echo frame analysis
- ECG encoder trains from scratch — it needs to learn cardiac-specific features
- Fusion layers, tokenizers, task-aware pooling, and both heads all train
- Result: ~4M trainable parameters (vs 27M in v2), dramatically reducing overfitting risk

In [ ]:
print("Partial freeze — ResNet50 frozen, ECG encoder + fusion trainable")

w_struct = compute_class_weights_binary(train_ds.labels_struct)
w_arr    = compute_class_weights_binary(train_ds.labels_arr)
print(f"Structural weights: normal={w_struct[0]:.3f}  disease={w_struct[1]:.3f}")
print(f"Arrhythmia weights: normal={w_arr[0]:.3f}  arrhythmia={w_arr[1]:.3f}")

crit_struct = FocalLoss(alpha=w_struct, gamma=2.0, label_smoothing=0.1)
crit_arr    = FocalLoss(alpha=w_arr,    gamma=2.0, label_smoothing=0.1)

model = CardiacFusionModel().to(DEVICE)
model.freeze_echo_encoder()
model.count_params()

## 6 · Training — End-to-End with Partial Freeze

**Differential learning rates:**
- ECG encoder: `LR × 1.0` — trained from scratch, needs full learning rate
- Fusion + tokenizers + heads: `LR × 2.0` — newest layers, trained most aggressively
- ResNet50: frozen (0 LR)

**Learning rate schedule:**
- 3-epoch linear warmup from `LR/10` → `LR`
- Cosine annealing after warmup → decays smoothly to near-zero

**Regularisation stack:**
- Focal loss (focus on hard examples) + class weights (up-weight minority)
- Weighted oversampling (balanced batches for structural task)
- Weight decay `1e-2` (strong L2 on all trainable params)
- Dropout `0.5` in MLP heads + label smoothing `0.1`
- Gradient clipping at norm `1.0` + Mixup `α=0.2`
- Early stopping patience=12

**What to watch:**
- Train accuracy should rise slowly (0.70–0.85 by epoch 30, NOT 0.98)
- Train–val gap should stay under 15%
- If structural improves but arrhythmia stalls → ECG encoder isn't learning → increase `LAMBDA_ARR`
- If arrhythmia improves but structural stalls → increase ResNet50 LR or unfreeze last layer

In [ ]:
s2_opt = torch.optim.AdamW([
    {'params': model.ecg_encoder.parameters(),               'lr': S2_LR * 1.0},
    {'params': list(model.phase_tok.parameters()) +
               list(model.cycle_tok.parameters()) +
               list(model.fusion.parameters())   +
               list(model.task_pool.parameters()) +
               list(model.head_structural.parameters()) +
               list(model.head_arrhythmia.parameters()) +
               [model.ecg_pos, model.echo_pos],              'lr': S2_LR * 2.0},
], weight_decay=1e-2)

WARMUP_EPOCHS = 3
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS * 0.1
    return 1.0

warmup_sched = torch.optim.lr_scheduler.LambdaLR(s2_opt, lr_lambda)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(s2_opt, T_max=S2_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
scaler       = torch.amp.GradScaler() if DEVICE.type == 'cuda' else None
ckpt_best    = os.path.join(OUTPUT_DIR, 'multitask_v3_best.pt')

best_combined_f1 = 0.0
no_improve       = 0
history = {'train_loss':[], 'val_loss':[],
           'train_s_acc':[], 'val_s_acc':[],
           'train_a_acc':[], 'val_a_acc':[]}

print(f"\n{'='*70}")
print(f" Multi-task v3: Structural + Arrhythmia (binary x binary)")
print(f" ResNet50 FROZEN | ECG encoder + fusion TRAINABLE (~4M params)")
print(f" Learned tokenizer ({NUM_ECG_TOKENS} ECG tokens) + Task-aware pooling")
print(f" Loss = {LAMBDA_STRUCT}*structural + {LAMBDA_ARR}*arrhythmia")
print(f" Up to {S2_EPOCHS} epochs | LR={S2_LR} | Patience={PATIENCE}")
print(f"{'='*70}")

for ep in range(1, S2_EPOCHS + 1):
    t0 = time.time()
    tl, tsa, taa = run_epoch_mt(model, train_loader, crit_struct, crit_arr, s2_opt, scaler, use_mixup=True)
    vl, vsa, vaa = run_epoch_mt(model, val_loader,   crit_struct, crit_arr)

    if ep <= WARMUP_EPOCHS:
        warmup_sched.step()
    else:
        cosine_sched.step()

    sf1, af1, combined_f1 = compute_combined_f1(model, val_loader)

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_s_acc'].append(tsa); history['val_s_acc'].append(vsa)
    history['train_a_acc'].append(taa); history['val_a_acc'].append(vaa)

    flag = ''
    if combined_f1 > best_combined_f1:
        best_combined_f1 = combined_f1
        no_improve = 0
        torch.save(model.state_dict(), ckpt_best)
        flag = '  <- BEST'
    else:
        no_improve += 1

    cur_lr = s2_opt.param_groups[-1]['lr']
    print(f"Ep {ep:03d}/{S2_EPOCHS} | "
          f"tr {tl:.3f} s={tsa:.3f} a={taa:.3f} | "
          f"val {vl:.3f} s={vsa:.3f} a={vaa:.3f} | "
          f"F1: s={sf1:.3f} a={af1:.3f} avg={combined_f1:.3f}{flag} | "
          f"lr={cur_lr:.2e} | {time.time()-t0:.1f}s")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {ep}")
        break

print(f"\nBest combined val F1: {best_combined_f1:.4f}")
plot_history_mt(history, 'MultiTask_v3')

## 7 · Test Set Evaluation

Loads best checkpoint and evaluates on held-out test set.

**Metrics for each task:**
- **AUROC** — threshold-independent discriminative ability. Target: ≥ 0.80
- **AUPRC** — area under precision-recall curve (more informative for imbalanced data)
- **Per-class precision/recall** — critical for minority class performance
- **Optimal threshold** — found on validation set, applied to test set for improved minority-class recall

**Visualisations:** confusion matrices + ROC curves for both tasks side by side

In [ ]:
model.load_state_dict(torch.load(ckpt_best, map_location=DEVICE, weights_only=True))

s_true, s_probs, a_true, a_probs = evaluate_mt(model, test_loader)

# ── Find optimal thresholds on validation set ─────────────────────────────────
val_s_true, val_s_probs, val_a_true, val_a_probs = evaluate_mt(model, val_loader)

def find_optimal_threshold(true, probs):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.2, 0.8, 0.01):
        f = f1_score(true, (probs >= t).astype(int), average='macro', zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

thresh_s = find_optimal_threshold(val_s_true, val_s_probs)
thresh_a = find_optimal_threshold(val_a_true, val_a_probs)
print(f"Optimal thresholds — structural: {thresh_s:.2f}  arrhythmia: {thresh_a:.2f}")

s_preds = (s_probs >= thresh_s).astype(int)
a_preds = (a_probs >= thresh_a).astype(int)

s_auroc = roc_auc_score(s_true, s_probs)
a_auroc = roc_auc_score(a_true, a_probs)
s_auprc = average_precision_score(s_true, s_probs)
a_auprc = average_precision_score(a_true, a_probs)
s_f1    = f1_score(s_true, s_preds, average='macro')
a_f1    = f1_score(a_true, a_preds, average='macro')

print(f"\n{'='*70}")
print(f"TEST SET RESULTS — Multi-Task Fusion v3")
print(f"{'='*70}")

print(f"\n--- STRUCTURAL DISEASE (threshold={thresh_s:.2f}) ---")
print(f"  AUROC : {s_auroc:.4f}")
print(f"  AUPRC : {s_auprc:.4f}")
print(f"  F1    : {s_f1:.4f}")
print(classification_report(s_true, s_preds, target_names=STRUCT_CLASSES, digits=3))

print(f"--- ARRHYTHMIA (threshold={thresh_a:.2f}) ---")
print(f"  AUROC : {a_auroc:.4f}")
print(f"  AUPRC : {a_auprc:.4f}")
print(f"  F1    : {a_f1:.4f}")
print(classification_report(a_true, a_preds, target_names=ARR_CLASSES, digits=3))

# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, true, preds, names, title in [
    (axes[0], s_true, s_preds, STRUCT_CLASSES, 'Structural Disease'),
    (axes[1], a_true, a_preds, ARR_CLASSES,    'Arrhythmia')
]:
    cm = confusion_matrix(true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_multitask_v3.png'), dpi=150)
plt.show()

# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, true, probs, auroc, title in [
    (axes[0], s_true, s_probs, s_auroc, 'Structural Disease'),
    (axes[1], a_true, a_probs, a_auroc, 'Arrhythmia')
]:
    fpr, tpr, _ = roc_curve(true, probs)
    ax.plot(fpr, tpr, 'b-', lw=2, label=f'AUROC = {auroc:.3f}')
    ax.plot([0,1], [0,1], 'k--', lw=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title(f'ROC — {title}')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_multitask_v3.png'), dpi=150)
plt.show()

print(f"\n{'='*70}")
print(f"COMBINED: AUROC={(s_auroc+a_auroc)/2:.4f}  F1={(s_f1+a_f1)/2:.4f}")
print(f"{'='*70}")

## 8 · XAI — Electromechanical Attribution Maps (EAM)

Visualises the cross-attention matrix from the last cross-attention layer.

**v3 change:** The attention matrix is now **4×3** (4 learned ECG tokens × 3 echo cycle tokens) instead of 3×3. Since the ECG tokens are learned (not hardcoded P/QRS/T), they're labelled T1–T4.

**How to read:**
- Rows = ECG tokens (T1, T2, T3, T4 — learned temporal segments)
- Columns = Echo tokens (End-Diastole, Mid, End-Systole)
- High value = model considers these two features informationally linked

**Also shows task-aware pooling weights** — which of the 7 fused tokens each task head attends to. Expect structural to weight echo tokens higher (tokens 5–7) and arrhythmia to weight ECG tokens higher (tokens 1–4).

Green title = correct prediction, Red = incorrect.

In [ ]:
ECG_TOK_NAMES  = [f'ECG-T{i+1}' for i in range(NUM_ECG_TOKENS)]
ECHO_TOK_NAMES = ['End-Diastole', 'Mid', 'End-Systole']
ALL_TOK_NAMES  = ECG_TOK_NAMES + ECHO_TOK_NAMES

def plot_eam_v3(model, loader, num_samples=6):
    model.eval()
    collected = []
    use_amp = DEVICE.type == 'cuda'
    with torch.no_grad():
        for ecg, echo, labels, _ in loader:
            if use_amp:
                with torch.autocast('cuda', torch.float16):
                    logits, _ = model(ecg.to(DEVICE), echo.to(DEVICE))
            else:
                logits, _ = model(ecg.to(DEVICE), echo.to(DEVICE))
            preds = logits.argmax(1).cpu()
            w = model.fusion.attn_weights['ecg_to_echo'].float()

            # Get task-aware pooling weights
            ecg_feat  = model.ecg_encoder(ecg.to(DEVICE))
            echo_feat = model.echo_encoder(echo.to(DEVICE))
            ecg_tok   = model.phase_tok(ecg_feat) + model.ecg_pos
            echo_tok  = model.cycle_tok(echo_feat) + model.echo_pos
            fused     = model.fusion(ecg_tok, echo_tok)
            _, _, s_w, a_w = model.task_pool(fused)

            for i in range(min(len(labels), num_samples - len(collected))):
                collected.append(dict(
                    attn     = w[i].numpy(),
                    s_pool_w = s_w[i].cpu().numpy(),
                    a_pool_w = a_w[i].cpu().numpy(),
                    true     = STRUCT_CLASSES[labels[i]],
                    pred     = STRUCT_CLASSES[preds[i]],
                    correct  = (labels[i] == preds[i]).item()
                ))
            if len(collected) >= num_samples:
                break

    # Plot cross-attention maps
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ax, s in zip(axes.flat, collected[:6]):
        im = ax.imshow(s['attn'], vmin=0, vmax=s['attn'].max(), cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(3)); ax.set_xticklabels(ECHO_TOK_NAMES, fontsize=8)
        ax.set_yticks(range(NUM_ECG_TOKENS)); ax.set_yticklabels(ECG_TOK_NAMES, fontsize=8)
        ax.set_xlabel('Echo token'); ax.set_ylabel('ECG token')
        color = 'green' if s['correct'] else 'red'
        ax.set_title(f"True: {s['true']}\nPred: {s['pred']}", color=color, fontsize=9, fontweight='bold')
        for r in range(NUM_ECG_TOKENS):
            for c in range(3):
                ax.text(c, r, f"{s['attn'][r,c]:.2f}", ha='center', va='center', fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.04)
    plt.suptitle('Electromechanical Attribution Maps — ECG→Echo cross-attention (v3)', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'eam_v3.png'), dpi=150)
    plt.show()

    # Plot task-aware pooling weights
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    avg_s = np.mean([s['s_pool_w'] for s in collected], axis=0)
    avg_a = np.mean([s['a_pool_w'] for s in collected], axis=0)
    for ax, weights, title in [
        (axes[0], avg_s, 'Structural Head — Token Weights'),
        (axes[1], avg_a, 'Arrhythmia Head — Token Weights')
    ]:
        colors = ['#3498db']*NUM_ECG_TOKENS + ['#e74c3c']*3  # blue=ECG, red=Echo
        ax.bar(range(len(weights)), weights, color=colors)
        ax.set_xticks(range(len(weights)))
        ax.set_xticklabels(ALL_TOK_NAMES, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Attention weight')
        ax.set_title(title)
        ax.axhline(y=1/len(weights), color='gray', linestyle='--', alpha=0.5, label='Uniform')
        ax.legend()
    plt.suptitle('Task-Aware Pooling — which tokens each head attends to', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'task_aware_pooling_v3.png'), dpi=150)
    plt.show()


plot_eam_v3(model, test_loader, num_samples=6)

## 9 · Confidence Calibration

A model that predicts "90% disease" should be correct ~90% of the time. If it's actually correct only 60% of the time, the model is **overconfident** — dangerous in clinical settings.

**Temperature scaling** is the standard post-hoc calibration method:
- Learn a single temperature parameter `T` on the validation set
- Divide logits by `T` before softmax: `calibrated_prob = softmax(logits / T)`
- `T > 1` → softer probabilities (fixes overconfidence)
- `T < 1` → sharper probabilities (fixes underconfidence)
- `T = 1` → no change (already calibrated)

**Reliability diagram** — plots predicted probability vs actual fraction of positives across 10 bins. A perfectly calibrated model follows the diagonal. Points above = underconfident, below = overconfident.

**Note:** Calibration does NOT change AUROC or accuracy — it only makes the probability values trustworthy without changing the ranking of predictions.

In [ ]:
from sklearn.calibration import calibration_curve


class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


def calibrate_model(model, val_loader, task='structural'):
    temp = TemperatureScaling().to(DEVICE)
    optimizer = torch.optim.LBFGS([temp.temperature], lr=0.01, max_iter=50)
    criterion = nn.CrossEntropyLoss()

    all_logits, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for ecg, echo, lbl_s, lbl_a in val_loader:
            out_s, out_a = model(ecg.to(DEVICE), echo.to(DEVICE))
            logits = out_s if task == 'structural' else out_a
            labels = lbl_s if task == 'structural' else lbl_a
            all_logits.append(logits)
            all_labels.append(labels.to(DEVICE))
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    def closure():
        optimizer.zero_grad()
        loss = criterion(temp(all_logits), all_labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    print(f"  {task} optimal temperature: {temp.temperature.item():.3f}")
    return temp


# ── Calibrate both tasks ──────────────────────────────────────────────────────
print("Learning calibration temperatures on validation set...")
temp_struct = calibrate_model(model, val_loader, 'structural')
temp_arr    = calibrate_model(model, val_loader, 'arrhythmia')

# ── Get calibrated test probabilities ─────────────────────────────────────────
model.eval()
s_logits_all, a_logits_all = [], []
s_true_cal, a_true_cal = [], []
with torch.no_grad():
    for ecg, echo, lbl_s, lbl_a in test_loader:
        out_s, out_a = model(ecg.to(DEVICE), echo.to(DEVICE))
        s_logits_all.append(out_s)
        a_logits_all.append(out_a)
        s_true_cal.extend(lbl_s.tolist())
        a_true_cal.extend(lbl_a.tolist())

s_logits_cal = temp_struct(torch.cat(s_logits_all))
a_logits_cal = temp_arr(torch.cat(a_logits_all))
s_probs_cal = torch.softmax(s_logits_cal, dim=1)[:, 1].cpu().numpy()
a_probs_cal = torch.softmax(a_logits_cal, dim=1)[:, 1].cpu().numpy()
s_true_cal = np.array(s_true_cal)
a_true_cal = np.array(a_true_cal)

print(f"\nCalibrated AUROC:")
print(f"  Structural: {roc_auc_score(s_true_cal, s_probs_cal):.4f}")
print(f"  Arrhythmia: {roc_auc_score(a_true_cal, a_probs_cal):.4f}")

# ── Reliability diagrams: before vs after calibration ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, true_arr, probs_before, probs_after, name in [
    (axes[0], s_true, s_probs, s_probs_cal, 'Structural'),
    (axes[1], a_true, a_probs, a_probs_cal, 'Arrhythmia')
]:
    try:
        frac_b, mean_b = calibration_curve(true_arr, probs_before, n_bins=10, strategy='uniform')
        ax.plot(mean_b, frac_b, 's--', color='#e74c3c', label='Before calibration', markersize=8)
    except:
        pass
    try:
        frac_a, mean_a = calibration_curve(true_arr, probs_after, n_bins=10, strategy='uniform')
        ax.plot(mean_a, frac_a, 'o-', color='#2ecc71', label='After calibration', markersize=8)
    except:
        pass
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.set_xlabel('Predicted probability', fontsize=11)
    ax.set_ylabel('Actual fraction of positives', fontsize=11)
    ax.set_title(f'{name}', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.suptitle('Confidence Calibration — Before vs After Temperature Scaling', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_v3.png'), dpi=150)
plt.show()

## 10 · Save Final Checkpoint

Saves model weights, calibration temperatures, architecture config, and training history. Self-contained — reload anywhere with just this file and the model class definitions.

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'cardiac_fusion_v3_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'struct_classes'  : STRUCT_CLASSES,
    'arr_classes'     : ARR_CLASSES,
    'temp_struct'     : temp_struct.temperature.item(),
    'temp_arr'        : temp_arr.temperature.item(),
    'optimal_thresholds': {'structural': thresh_s, 'arrhythmia': thresh_a},
    'config': dict(
        ecg_base_dim       = ECG_BASE_DIM,
        echo_base_dim      = ECHO_BASE_DIM,
        proj_dim           = PROJ_DIM,
        num_ecg_tokens     = NUM_ECG_TOKENS,
        attn_heads         = ATTN_HEADS,
        attn_layers        = ATTN_LAYERS,
        num_classes_struct = NUM_CLASSES_STRUCT,
        num_classes_arr    = NUM_CLASSES_ARR,
        lambda_struct      = LAMBDA_STRUCT,
        lambda_arr         = LAMBDA_ARR,
    ),
    'history': history,
    'test_results': {
        's_auroc': s_auroc, 'a_auroc': a_auroc,
        's_auprc': s_auprc, 'a_auprc': a_auprc,
        's_f1': s_f1, 'a_f1': a_f1,
    }
}, final_path)

print(f"Saved: {final_path}")
print(f"\nAll output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:45s}  {mb:7.1f} MB")